# Project 2: Linear vs. non-linear on biological data

Follow-on from Project 1. Project 1 showed that a linear model fails and an MLP succeeds on `make_circles` — clean, synthetic, low-dimensional data. This project asks the same linear-vs-non-linear question on **real biological data**, where the answer is not guaranteed to come out the same way.

In this project, I use the seminal **Golub et al. (1999)** leukemia gene expression dataset. This paper was groundbreaking: it proved for the first time that cancer could be classified purely by analyzing gene expression profiles, without relying on prior biological knowledge.

**Modern Relevance (p >> n):**
Even today, we frequently encounter $p \gg n$ datasets in bioinformatics, where features vastly outnumber samples. For example:
1. **Rare Disease Biomarker Discovery:** Sequencing 20,000 genes for a cohort of only 15 patients.
2. **Personalized Medicine Clinical Trials:** Gathering high-resolution proteomic profiles for 50 patients.

In these scenarios, complex deep learning models are prone to memorize the tiny sample set instead of learning biological signals. Testing the "neural nets are needed for non-linear biological data" claim requires a case where it's allowed to fail.

## Step 1: Import libraries

First, I import the necessary libraries for data processing, machine learning pipelines, and visualization.

In [ ]:
# Import JSON for saving output metrics\nimport json\n# Pathlib for cross-platform file paths\nfrom pathlib import Path\n\n# NumPy and Pandas for data manipulation\nimport numpy as np\nimport pandas as pd\n# Matplotlib for generating data visualizations\nimport matplotlib.pyplot as plt\n\n# Scikit-Learn tools for PCA (dimensionality reduction) and Data Scaling\nfrom sklearn.decomposition import PCA\nfrom sklearn.preprocessing import StandardScaler\n# Tools for feature selection based on statistical significance (ANOVA)\nfrom sklearn.feature_selection import SelectKBest, f_classif\n# Pipeline to chain preprocessing and modeling steps\nfrom sklearn.pipeline import Pipeline\n# Tools for cross-validation\nfrom sklearn.model_selection import StratifiedKFold, cross_val_score\n# Linear classification models\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC, LinearSVC\n# Non-linear Neural Network classification model\nfrom sklearn.neural_network import MLPClassifier\n# Dummy baseline model for comparison\nfrom sklearn.dummy import DummyClassifier\n\n# Define and create a directory to save our output figures and metrics\nRESULTS_DIR = Path("results")\nRESULTS_DIR.mkdir(exist_ok=True)\n# Define a random seed to ensure our results are exactly reproducible\nRANDOM_STATE = 0\

## Step 2: Load Data and Establish Baseline

I load the cleaned dataset. It contains 72 patients (our samples) and 3,571 genes (our features), labeled as either ALL or AML leukemia.

Before training any model, I establish a 'dummy' baseline. This model simply guesses the most frequent class (ALL) every single time. It achieves about 65% accuracy. This establishes the absolute minimum floor that a real model has to beat to prove it is learning actual biological signals.

In [ ]:
# Define the path to our cleaned leukemia dataset\nDATA_PATH = Path("data/leukemia_clean.csv")\n\n# Ensure the file exists before proceeding\nif not DATA_PATH.exists():\n    raise FileNotFoundError(\n        f"{DATA_PATH} not found. Run `python data/download_data.py` from this "\n        "project folder first to download and prepare the dataset."\n    )\n\n# Load the CSV data into a Pandas DataFrame\ndf = pd.read_csv(DATA_PATH)\n# Extract the features (genes) by dropping the label column\nX = df.drop(columns=["label"]).values\n# Convert the textual labels into binary numerical targets (AML = 1, ALL = 0)\ny = (df["label"] == "AML").astype(int).values\n\n# Print out the shape of our dataset for verification\nprint(f"Loaded {X.shape[0]} samples, {X.shape[1]} genes.")\n# Print the exact class distribution\nprint(f"Class balance: ALL={np.sum(y==0)}, AML={np.sum(y==1)}")\

## Step 3: PCA to 2D for visual inspection

Because of human limitations, we can only visually perceive up to 3 dimensions. We cannot plot 3,571 genes on a graph.

To overcome this, I use Principal Component Analysis (PCA) to compress all 3,571 genes down to 2 dimensions. PCA is a standard, deterministic mathematical projection that captures the maximum variance in the data without requiring hyperparameter tuning (unlike t-SNE or UMAP). 

I do this purely for a visual sanity check to see if the ALL and AML patients naturally cluster apart based on their gene expressions. I do not train the models on this 2D data.

**PCA Projection Interpretation:**

![PCA Projection](results/pca_projection.png)

Looking at the resulting PCA projection plot, we can see that the ALL (blue) and AML (orange) patients form somewhat distinct clusters, but there is noticeable overlap in the center. The first two principal components capture the highest variance across all 3,571 genes.

This tells us two things:
1. There is a genuine, strong biological signal separating the two leukemias (the points aren't completely random noise).
2. The classes are not trivially separable with a single straight line in 2D space. A machine learning model will need to leverage the higher-dimensional gene interactions to accurately separate the patients.


In [ ]:
# Initialize a naive model that only ever predicts the majority class\ndummy = DummyClassifier(strategy="most_frequent")\n# Setup a 5-fold Stratified Cross-Validation to maintain the ALL/AML ratio across folds\ncv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n# Calculate the accuracy of the naive model across all 5 folds\ndummy_scores = cross_val_score(dummy, X, y, cv=cv, scoring="accuracy")\n\n# Print the mean accuracy and standard deviation of our baseline\nprint(f"Majority-class baseline: {dummy_scores.mean():.3f} +/- {dummy_scores.std():.3f}")\

## Step 4: Setup Pipeline and CV

Now, I compare a suite of linear models (Logistic Regression, Linear SVM) against non-linear models (RBF SVM, MLP Neural Network).

**Identical Preprocessing & Information Leaks**
To ensure a fair, apples-to-apples comparison, I use a scikit-learn `Pipeline`. This guarantees that every model receives the exact same standard scaling and feature selection steps (keeping the top 50 genes via ANOVA F-test). We are comparing the models themselves, not the way the data was prepared for them.

Crucially, as taught in standard Machine Learning curriculums (such as the Kaggle Intro to ML course), feature selection MUST happen *inside* the cross-validation loop. If I filtered the top 50 genes using the entire 72 patients before splitting the data, the training phase would get an unfair 'sneak peek' at the validation cohort. This is a classic information leak that artificially inflates accuracy.

**5-Fold Cross-Validation**
Because I only have 72 samples, a single train/validation split would leave only ~21 patients for testing—far too few for a reliable metric. Instead, I use 5-fold cross validation. This splits the data into 5 chunks, training on 4 and testing on 1, rotating until every patient has been in the unseen validation cohort exactly once.

In [ ]:
# Standardize the features so that genes with larger expression scales don't dominate PCA\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(X)\n\n# Initialize a PCA model to compress the 3571 genes down to just 2 Principal Components\npca = PCA(n_components=2, random_state=RANDOM_STATE)\n# Fit PCA on the scaled data and transform it to 2D\nX_pca = pca.fit_transform(X_scaled)\n\n# Create a matplotlib figure to plot our 2D projection\nfig, ax = plt.subplots(figsize=(6, 5))\n# Iterate over both classes (0: ALL, 1: AML) to plot them in different colors\nfor label, name, color in [(0, "ALL", "tab:blue"), (1, "AML", "tab:orange")]:\n    # Create a boolean mask to isolate patients of the current class\n    mask = y == label\n    # Scatter plot the Principal Component 1 vs Principal Component 2 for this class\n    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, alpha=0.7, color=color)\n\n# Label the axes with the percentage of variance explained by each principal component\nax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")\nax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")\nax.set_title("PCA projection: ALL vs. AML gene expression")\nax.legend()\nfig.tight_layout()\n\n# Save the plot to the results folder\nfig.savefig(RESULTS_DIR / "pca_projection.png", dpi=150)\nplt.show()\

In [ ]:
# Define the number of top statistically significant genes we want to keep\nN_FEATURES = 50  \n\n# Helper function to generate a machine learning pipeline\ndef make_pipeline(clf):\n    return Pipeline([\n        # Step 1: Standardize features (mean=0, variance=1)\n        ("scale", StandardScaler()),\n        # Step 2: Select the top 50 genes using the ANOVA F-value metric\n        ("select", SelectKBest(score_func=f_classif, k=N_FEATURES)),\n        # Step 3: Train the actual classification model passed to this function\n        ("clf", clf),\n    ])\

## Step 5: Linear vs. non-linear models, compared with 5-fold cross-validation

72 samples is too small to trust a single train/test split (Project 1 used
one split because `make_circles` is easy; here we can't get away with that).
We use stratified 5-fold CV and report mean +/- std accuracy for each model,
all built on the *same* feature-selection pipeline so the comparison is fair.


In [ ]:
# Define a dictionary of models to test, mixing linear and non-linear types\nmodels = {\n    # Highly regularized linear models\n    "Logistic Regression (linear)": LogisticRegression(max_iter=5000),\n    "Linear SVM": LinearSVC(max_iter=5000),\n    # Non-linear models using radial basis functions and neural networks\n    "RBF SVM (non-linear)": SVC(kernel="rbf"),\n    "MLP, 1 hidden layer of 10 (non-linear)": MLPClassifier(\n        hidden_layer_sizes=(10,), max_iter=3000, random_state=RANDOM_STATE\n    ),\n}\n\n# Dictionary to store the cross-validation scores for later comparison\ncv_results = {}\n\n# Iterate over each model definition\nfor name, clf in models.items():\n    # Build the complete pipeline (scale -> select -> model)\n    pipe = make_pipeline(clf)\n    # Evaluate the pipeline using 5-fold cross-validation\n    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")\n    \n    # Store the mean, standard deviation, and individual fold scores\n    cv_results[name] = {"mean": scores.mean(), "std": scores.std(), "folds": scores.tolist()}\n    # Print the model's performance to the console\n    print(f"{name:45s}  {scores.mean():.3f} +/- {scores.std():.3f}   (folds: {np.round(scores,3)})")\

## Step 6: Plot the comparison

I visualize the cross-validation accuracy of the models using a bar chart.

![Model Comparison](results/model_comparison.png)

As anticipated, in this high-dimensional $p \gg n$ setting, the linear models (Logistic Regression, Linear SVM) perform just as well as, or slightly better than, the complex non-linear models (MLP). The neural network shows higher variance (wider error bars) and lower mean accuracy because its excess capacity makes it prone to overfitting the 72 samples.

This directly mirrors the significance of Golub et al.'s original 1999 findings: linear boundaries are highly effective for gene expression classification. Even today, non-linearity is not a silver bullet in bioinformatics.


In [ ]:
# Create a figure for a horizontal bar chart\nfig, ax = plt.subplots(figsize=(7, 4))\n# Extract model names and their corresponding mean accuracies/standard deviations\nnames = list(cv_results.keys())\nmeans = [cv_results[n]["mean"] for n in names]\nstds = [cv_results[n]["std"] for n in names]\n\n# Plot horizontal bars representing the mean accuracy, with error bars representing std dev\nax.barh(names, means, xerr=stds, color="tab:blue", alpha=0.8)\n# Draw a vertical dashed red line to represent the baseline accuracy\nax.axvline(dummy_scores.mean(), color="red", linestyle="--",\n           label=f"Majority-class baseline ({dummy_scores.mean():.2f})")\n\n# Format the plot labels and limits\nax.set_xlabel("5-fold CV accuracy")\nax.set_xlim(0, 1.05)\nax.set_title(f"Linear vs. non-linear models on Golub leukemia data (top {N_FEATURES} genes)")\nax.legend(loc="lower right")\nfig.tight_layout()\n\n# Save the final comparison figure to disk\nfig.savefig(RESULTS_DIR / "model_comparison.png", dpi=150)\nplt.show()\

## Step 7: Save all results to disk

Why is this step important? In any data science pipeline, generating a plot in a Jupyter notebook isn't enough. We must serialize the exact hyperparameter configurations, the cross-validation splits, and the final metrics to a JSON file on disk. This ensures that the experiment is fully reproducible for future reference and that these metrics can be programmatically compared against future models without having to rerun the entire training pipeline.

In [ ]:
# Construct a detailed summary dictionary containing all experiment metadata\nsummary = {\n    "dataset": {\n        "name": "Golub et al. 1999 leukemia gene expression (ALL vs AML)",\n        "source_url": "http://hastie.su.domains/CASI_files/DATA/leukemia_small.csv",\n        "n_samples": int(X.shape[0]),\n        "n_genes_total": int(X.shape[1]),\n        "n_genes_selected_per_fold": N_FEATURES,\n        "class_counts": {"ALL": int(np.sum(y == 0)), "AML": int(np.sum(y == 1))},\n    },\n    "cv": {"n_splits": 5, "shuffle": True, "random_state": RANDOM_STATE},\n    "majority_class_baseline_accuracy": {\n        "mean": float(dummy_scores.mean()), "std": float(dummy_scores.std())\n    },\n    "model_comparison": cv_results,\n}\n\n# Serialize the summary dictionary into a nicely formatted JSON file on disk\nwith open(RESULTS_DIR / "metrics.json", "w") as f:\n    json.dump(summary, f, indent=2)\n\n# Print out the final JSON to the notebook for inspection\nprint(json.dumps(summary, indent=2))\

## Step 8: Final Conclusion

By comparing the models on the Golub 1999 dataset, we successfully demonstrated that **non-linearity is not a silver bullet in machine learning.** 

While Project 1 proved that a simple linear model fails catastrophically on the 2D `make_circles` dataset, this project proves the opposite for biological data. When faced with high-dimensional data ($p \gg n$), the excess capacity of the Multi-Layer Perceptron (neural network) caused it to overfit the small sample size (72 patients), resulting in higher variance across folds.

Conversely, the simpler, highly-constrained linear models (Logistic Regression and Linear SVM) were able to robustly capture the true biological signal, generalizing better and achieving higher mean accuracy. 

**Key Takeaway:** Always match the complexity of your model to the geometry and dimensionality of your data. For many high-dimensional bioinformatics tasks, a heavily regularized linear model remains the gold standard.